# Cancer Detection using CNN — Histopathological Images

| | |
|---|---|
| **Name** | Ayush Ranjan |
| **Roll No** | 23MIP10135 |
| **Institute** | VIT Bhopal |
| **Dataset** | [Lung and Colon Cancer Histopathological Images](https://www.kaggle.com/datasets/andrewmvd/lung-and-colon-cancer-histopathological-images) — Kaggle |

**Objective:** Detect and classify cancer types from histopathology slides using a deep CNN. 
**Classes:** Lung Adenocarcinoma (aca), Lung Benign (n), Lung Squamous Cell Carcinoma (scc), 
Colon Adenocarcinoma (aca), Colon Benign (n) 
**Total Images:** 25,000 (5,000 per class at 768×768, resized to 150×150 for training)


## Step 1: Install Libraries


In [ ]:
!pip install kaggle tensorflow scikit-learn matplotlib seaborn pillow -q
print('Libraries ready!')

## Step 2: Kaggle API Setup
> Run this cell — it configures Kaggle credentials automatically.


In [ ]:
import os

# ── Kaggle API Token ──
os.environ['KAGGLE_KEY'] = 'KGAT_a8751a0f54996277f33924e203785e29'

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    import json
    json.dump({"key": "KGAT_a8751a0f54996277f33924e203785e29"}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API configured!')

## Step 3: Download Dataset


In [ ]:
!kaggle datasets download -d andrewmvd/lung-and-colon-cancer-histopathological-images \
    -p /content/cancer_raw --unzip -q
print('Downloaded!')
!ls /content/cancer_raw/
!ls /content/cancer_raw/lung_colon_image_set/

## Step 4: Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, shutil, random, warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Dense, Dropout,
                                      BatchNormalization, GlobalAveragePooling2D)
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(42); tf.random.set_seed(42); random.seed(42)
print(f'TensorFlow {tf.__version__} | GPU: {tf.config.list_physical_devices("GPU")}')

## Step 5: Explore Dataset


In [ ]:
base_dir = '/content/cancer_raw/lung_colon_image_set'
all_classes = sorted([d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir,d))])
print('Classes:', all_classes)

class_counts = {}
for cls in all_classes:
    p = os.path.join(base_dir, cls)
    n = len([f for f in os.listdir(p) if f.endswith(('.jpg','.jpeg','.png'))])
    class_counts[cls] = n
    print(f'  {cls}: {n} images')
print(f'Total: {sum(class_counts.values())} images')

In [ ]:
# Bar chart
colors=['#E74C3C','#2ECC71','#E67E22','#3498DB','#9B59B6']
plt.figure(figsize=(12,5))
bars=plt.bar(class_counts.keys(),class_counts.values(),color=colors,edgecolor='k',lw=0.8)
for bar,v in zip(bars,class_counts.values()):
    plt.text(bar.get_x()+bar.get_width()/2,bar.get_height()+30,str(v),ha='center',fontweight='bold',fontsize=11)
plt.title('Cancer Dataset — Class Distribution',fontsize=14,fontweight='bold')
plt.xlabel('Class'); plt.ylabel('Images'); plt.xticks(rotation=20,ha='right')
plt.tight_layout(); plt.savefig('cancer_class_dist.png',dpi=150,bbox_inches='tight'); plt.show()

# Sample images
fig,axes=plt.subplots(1,5,figsize=(20,4)); 
for i,cls in enumerate(all_classes):
    p=os.path.join(base_dir,cls); img=load_img(os.path.join(p,os.listdir(p)[0]),target_size=(150,150))
    axes[i].imshow(img); axes[i].set_title(cls.replace('_','\n'),fontsize=9,fontweight='bold'); axes[i].axis('off')
plt.suptitle('Cancer Histopathology — Sample Images',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.savefig('cancer_samples.png',dpi=150,bbox_inches='tight'); plt.show()

## Step 6: Train / Val / Test Split


In [ ]:
split_dir = '/content/cancer_split'
SPLITS    = {'train':0.70,'val':0.15,'test':0.15}

if not os.path.exists(split_dir):
    for s in SPLITS:
        for cls in class_counts: os.makedirs(os.path.join(split_dir,s,cls),exist_ok=True)
    for cls in class_counts:
        src  = os.path.join(base_dir,cls)
        imgs = [f for f in os.listdir(src) if f.endswith(('.jpg','.jpeg','.png'))]
        random.shuffle(imgs)
        n=len(imgs); n_tr=int(n*0.70); n_v=int(n*0.15)
        for f in imgs[:n_tr]:       shutil.copy(os.path.join(src,f),os.path.join(split_dir,'train',cls,f))
        for f in imgs[n_tr:n_tr+n_v]: shutil.copy(os.path.join(src,f),os.path.join(split_dir,'val',cls,f))
        for f in imgs[n_tr+n_v:]:   shutil.copy(os.path.join(src,f),os.path.join(split_dir,'test',cls,f))
    print('Split done!')
else: print('Already split.')

for s in SPLITS:
    tot=sum(len(os.listdir(os.path.join(split_dir,s,c))) for c in class_counts)
    print(f'{s}: {tot} images')

## Step 7: Data Augmentation & Generators


In [ ]:
IMG_SIZE=150; BATCH_SIZE=64; EPOCHS=50

train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=90,
    horizontal_flip=True, vertical_flip=True,
    zoom_range=0.15, width_shift_range=0.1, height_shift_range=0.1,
    brightness_range=[0.85,1.15], fill_mode='reflect'
)
other_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(split_dir,'train'), target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True, seed=42
)
val_gen = other_datagen.flow_from_directory(
    os.path.join(split_dir,'val'), target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
test_gen = other_datagen.flow_from_directory(
    os.path.join(split_dir,'test'), target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

NUM_CLASSES=len(train_gen.class_indices); CLASS_NAMES=list(train_gen.class_indices.keys())
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')
print(f'Train: {train_gen.n} | Val: {val_gen.n} | Test: {test_gen.n}')

## Step 8: Build Deep CNN


In [ ]:
def build_cancer_cnn(num_classes, input_shape=(150,150,3)):
    """Deep CNN for Cancer Detection — 5 conv blocks + GAP + Dense head"""
    model = Sequential([
        Conv2D(32,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4),input_shape=input_shape),
        BatchNormalization(), Conv2D(32,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), MaxPooling2D((2,2)), Dropout(0.2),

        Conv2D(64,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Conv2D(64,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), MaxPooling2D((2,2)), Dropout(0.25),

        Conv2D(128,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Conv2D(128,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Conv2D(128,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), MaxPooling2D((2,2)), Dropout(0.3),

        Conv2D(256,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Conv2D(256,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Conv2D(256,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), MaxPooling2D((2,2)), Dropout(0.35),

        Conv2D(512,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Conv2D(512,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), MaxPooling2D((2,2)), Dropout(0.35),

        GlobalAveragePooling2D(),
        Dense(1024,activation='relu',kernel_regularizer=l2(1e-4)), BatchNormalization(), Dropout(0.5),
        Dense(512,activation='relu',kernel_regularizer=l2(1e-4)), Dropout(0.4),
        Dense(num_classes, activation='softmax')
    ], name='CancerDetection_CNN')
    return model

model = build_cancer_cnn(NUM_CLASSES)
model.summary()
print(f'Total params: {model.count_params():,}')

## Step 9: Compile & Train


In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.TopKCategoricalAccuracy(k=2,name='top2_acc')]
)

callbacks=[
    EarlyStopping(monitor='val_accuracy',patience=15,restore_best_weights=True,verbose=1),
    ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=5,min_lr=1e-7,verbose=1),
    ModelCheckpoint('/content/best_cancer_cnn.keras',monitor='val_accuracy',save_best_only=True,verbose=1)
]

history=model.fit(
    train_gen, epochs=EPOCHS, validation_data=val_gen, callbacks=callbacks, verbose=1
)
print(f'Best Val Accuracy: {max(history.history["val_accuracy"])*100:.2f}%')

## Step 10: Results & Evaluation


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(20,6))
for ax,m,label in zip(axes,['accuracy','loss','auc'],['Accuracy','Loss','AUC']):
    ax.plot(history.history[m],'b-o',label='Train',lw=2,ms=4)
    ax.plot(history.history[f'val_{m}'],'r-o',label='Validation',lw=2,ms=4)
    ax.set_title(label,fontsize=13,fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel(label)
    ax.legend(); ax.grid(True,alpha=0.3)
plt.suptitle('Cancer Detection CNN — Training History',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.savefig('cancer_history.png',dpi=150,bbox_inches='tight'); plt.show()

test_gen.reset()
res=model.evaluate(test_gen,verbose=1)
print(f'Test Accuracy : {res[1]*100:.2f}%')
print(f'Test AUC      : {res[2]:.4f}')
print(f'Top-2 Accuracy: {res[3]*100:.2f}%')
print(f'Test Loss     : {res[0]:.4f}')

In [ ]:
# Confusion matrix
test_gen.reset()
y_pred=np.argmax(model.predict(test_gen,verbose=0),axis=1)
y_true=test_gen.classes
cm=confusion_matrix(y_true,y_pred)
fig,ax=plt.subplots(figsize=(12,9))
im=ax.imshow(cm,cmap='YlOrRd'); plt.colorbar(im)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels([c.replace('_','\n') for c in CLASS_NAMES],fontsize=10)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels([c.replace('_','\n') for c in CLASS_NAMES],fontsize=10)
thresh=cm.max()/2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=11,fontweight='bold',
                color='white' if cm[i,j]>thresh else 'black')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Cancer Detection CNN — Confusion Matrix',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.savefig('cancer_confusion_matrix.png',dpi=150,bbox_inches='tight'); plt.show()
print(classification_report(y_true,y_pred,target_names=CLASS_NAMES))

In [ ]:
# Sample predictions
test_gen.reset()
imgs,labels=next(test_gen)
preds=model.predict(imgs,verbose=0)
n=min(10,len(imgs))
fig,axes=plt.subplots(2,5,figsize=(20,9)); axes=axes.flatten()
for i in range(n):
    axes[i].imshow(imgs[i])
    ti=np.argmax(labels[i]); pi=np.argmax(preds[i]); conf=preds[i][pi]*100
    axes[i].set_title(f'True: {CLASS_NAMES[ti].replace("_"," ")}\nPred: {CLASS_NAMES[pi].replace("_"," ")} ({conf:.1f}%)',
                      color='green' if ti==pi else 'red',fontsize=8,fontweight='bold')
    axes[i].axis('off')
for i in range(n,len(axes)): axes[i].axis('off')
plt.suptitle('Cancer Detection — Sample Predictions',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.savefig('cancer_predictions.png',dpi=150,bbox_inches='tight'); plt.show()

## Summary


In [ ]:
print('='*55)
print('    CANCER DETECTION CNN — SUMMARY')
print('='*55)
print('  Name        : Ayush Ranjan')
print('  Roll No     : 23MIP10135')
print('  Institute   : VIT Bhopal')
print('  Dataset     : Lung & Colon Cancer Histopathology (Kaggle)')
print(f'  Classes     : {NUM_CLASSES}')
print(f'  Class Names : {", ".join(CLASS_NAMES)}')
print(f'  Image Size  : {IMG_SIZE}x{IMG_SIZE}')
print('  Architecture: Deep CNN — 5 conv blocks + GAP')
print(f'  Parameters  : {model.count_params():,}')
print(f'  Test Accuracy: {res[1]*100:.2f}%')
print(f'  Test AUC    : {res[2]:.4f}')
print('='*55)